# 🧠 LLM Data Preprocessing Pipeline — Multiple Datasets

This notebook walks through every major step for preprocessing raw text datasets for LLM training:

1. Data Collection & Inventory
2. Data Cleaning (per dataset)
3. Quality Filtering
4. Normalization
5. Tokenization
---

## 📦 Step 0: Install Dependencies

In [1]:
!pip install datasets transformers sentencepiece datasketch langdetect pandas pyarrow tqdm ftfy

In [2]:
import os
import re
import json
import hashlib
import unicodedata
import random
from collections import defaultdict

import pandas as pd
import numpy as np
from tqdm import tqdm
import ftfy

from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset
from datasketch import MinHash, MinHashLSH
from langdetect import detect, LangDetectException
from transformers import AutoTokenizer

print('✅ All libraries imported successfully')

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.0
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


✅ All libraries imported successfully


---
## 📂 Step 1: Data Collection & Inventory

Define your dataset sources, formats, and metadata. Here we simulate three datasets: web crawl, books, and code.

In [3]:
import os
import json

os.environ["DATA_DIR"] = "/workspace/data"

data_dir = os.environ.get("DATA_DIR", "./data")
DATASET_REGISTRY = {}

print("📂 Loading standard JSON files...")

if not os.path.isdir(data_dir):
    print(f"❌ Directory not found: {data_dir}")
else:
    for filename in os.listdir(data_dir):
        if filename.endswith(".json"):
            dataset_name = os.path.splitext(filename)[0]
            file_path = os.path.join(data_dir, filename)
            
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    raw_data = json.load(f)
                
                    if isinstance(raw_data, dict) and 'questions' in raw_data:
                        data_list = raw_data['questions']
                    elif isinstance(raw_data, list):
                        data_list = raw_data
                    else:
                        data_list = [raw_data]

                    DATASET_REGISTRY[dataset_name] = {
                        'data': data_list,
                    }
                except json.JSONDecodeError as e:
                    print(f"⚠️ Could not parse {filename}: {e}")

# Check results
print("\n📋 Dataset Inventory:")
for name, meta in DATASET_REGISTRY.items():
    print(f"  [{name}] — {len(meta['data'])} documents")

📂 Loading standard JSON files...

📋 Dataset Inventory:
  [13B1_golden] — 85 documents
  [13B2_golden] — 85 documents
  [13B3_golden] — 85 documents
  [13B4_golden] — 85 documents
  [training13b] — 5389 documents


---
## 🧹 Step 2: Data Cleaning

In [4]:
def remove_html_tags(text: str) -> str:
    """Strip HTML tags from text."""
    return re.sub(r'<[^>]+>', '', text)

def fix_encoding(text: str) -> str:
    """Fix mojibake and encoding issues using ftfy."""
    return ftfy.fix_text(text)

def is_low_quality(text: str, min_words: int = 5, max_symbol_ratio: float = 0.3) -> bool:
    """Flag text as low quality based on heuristics."""
    words = text.split()
    if len(words) < min_words:
        return True
    symbols = sum(1 for c in text if not c.isalnum() and not c.isspace())
    if symbols / max(len(text), 1) > max_symbol_ratio:
        return True
    return False

def detect_language(text: str, target_lang: str = "en") -> bool:
    """Return True if text matches the target language."""
    try:
        return detect(text) == target_lang
    except LangDetectException:
        return False

def remove_pii(text: str) -> str:
    """Remove common PII patterns (email, phone, SSN)."""
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.]+', '[EMAIL]', text)
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[PHONE]', text)
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[SSN]', text)
    return text


def clean_document(doc: dict) -> dict | None:
    """Full cleaning pipeline for a single document."""
    
    # 1. Identify the text field. 
    # In BioASQ 'body' is the question. 
    # If you want to clean snippets, you'll need to join them first.
    text = doc.get("body", "") 
    
    # If 'body' is empty, let's check for 'snippets'
    if not text and "snippets" in doc:
        text = " ".join([s.get("text", "") for s in doc["snippets"]])

    # 2. Run the cleaning steps
    text = remove_html_tags(text)
    text = fix_encoding(text)
    text = text.strip()
    
    if is_low_quality(text):
        return None
    if not detect_language(text):
        return None
    
    text = remove_pii(text)
    
    # 3. Return the doc with the NEW 'text' field (standardizing it for RAG)
    return {**doc, "text": text}

# Apply cleaning to all datasets
cleaned_datasets = {}
for name, meta in DATASET_REGISTRY.items():
    print(f"🧹 Cleaning dataset: {name} ({len(meta['data'])} documents)")
    cleaned_docs = []
    for doc in tqdm(meta['data'], desc=f"Cleaning {name}"):
        cleaned_doc = clean_document(doc)
        if cleaned_doc:
            cleaned_docs.append(cleaned_doc)
    cleaned_datasets[name] = cleaned_docs
    print(f"✅ Cleaned {len(cleaned_docs)} documents from {name}")

🧹 Cleaning dataset: 13B1_golden (85 documents)


Cleaning 13B1_golden: 100% 85/85 [00:00<00:00, 160.82it/s]


✅ Cleaned 73 documents from 13B1_golden
🧹 Cleaning dataset: 13B2_golden (85 documents)


Cleaning 13B2_golden: 100% 85/85 [00:00<00:00, 384.65it/s]


✅ Cleaned 74 documents from 13B2_golden
🧹 Cleaning dataset: 13B3_golden (85 documents)


...

✅ Cleaned 77 documents from 13B3_golden
🧹 Cleaning dataset: 13B4_golden (85 documents)


Cleaning 13B4_golden: 100% 85/85 [00:00<00:00, 475.56it/s]


✅ Cleaned 71 documents from 13B4_golden
🧹 Cleaning dataset: training13b (5389 documents)


Cleaning training13b: 100% 5389/5389 [00:10<00:00, 515.40it/s]

✅ Cleaned 4794 documents from training13b


---
## 🎯 Step 3: Quality Filtering

In [5]:
def avg_word_length(text: str) -> float:
    words = text.split()
    return sum(len(w) for w in words) / max(len(words), 1)

def bullet_density(text: str) -> float:
    lines = text.splitlines()
    bullet_lines = sum(1 for l in lines if l.strip().startswith(('•', '-', '*', '·')))
    return bullet_lines / max(len(lines), 1)

def passes_quality_filter(doc: dict, verbose: bool = False) -> bool:
    text = doc.get("text", "")
    if not text or len(text) < 20:
        if verbose: print(f"FAIL length: {len(text)}")
        return False
    awl = avg_word_length(text)
    if not (2.5 <= awl <= 15):
        if verbose: print(f"FAIL awl: {awl:.2f}")
        return False
    if bullet_density(text) > 0.6:
        if verbose: print(f"FAIL bullets")
        return False
    return True

quality_filtered = {}
for name, docs in cleaned_datasets.items():
    before = len(docs)
    filtered = [doc for doc in docs if passes_quality_filter(doc)]
    quality_filtered[name] = filtered
    print(f"[{name}] quality filter: {before} → {len(filtered)} docs")

[13B1_golden] quality filter: 73 → 73 docs
[13B2_golden] quality filter: 74 → 74 docs
[13B3_golden] quality filter: 77 → 77 docs
[13B4_golden] quality filter: 71 → 71 docs
[training13b] quality filter: 4794 → 4793 docs


---
## ✏️ Step 4: Text Normalization

In [6]:
def normalize_text(text: str, form: str = "NFC") -> str:
    """Unicode normalization + whitespace cleanup."""
    # Unicode normalization
    text = unicodedata.normalize(form, text)
    # Normalize whitespace
    text = re.sub(r'\r\n|\r', '\n', text)         # normalize line endings
    text = re.sub(r'\t', ' ', text)               # tabs → spaces
    text = re.sub(r' {2,}', ' ', text)            # collapse multiple spaces
    text = re.sub(r'\n{3,}', '\n\n', text)        # collapse excess blank lines
    text = text.strip()
    return text

normalized_datasets = {}
for name, docs in quality_filtered.items():
    normalized = [{**doc, "text": normalize_text(doc["text"])} for doc in docs]
    normalized_datasets[name] = normalized

training_raw_data = normalized_datasets["training13b"]
test_raw_data = normalized_datasets["13B1_golden"] + normalized_datasets["13B2_golden"] + normalized_datasets["13B3_golden"] + normalized_datasets["13B4_golden"]

print("✅ Normalization complete")

# Preview a sample
for name, docs in normalized_datasets.items():
    if docs:
        print(f"\n[{name}] sample: {docs[0]['text'][:120]}..." if len(docs[0]['text']) > 120 else f"\n[{name}] sample: {docs[0]['text']}")

✅ Normalization complete

[13B1_golden] sample: What proportion of colorectal cancer cases are not assignable to any of the consensus molecular subtype (CMS) groups?

[13B2_golden] sample: Which ensemble machine-learning framework has been developed harnessing UK biobank data?

[13B3_golden] sample: How many primary genetic associations were identified through pQTL mapping within the Pharma Proteomics Project?

[13B4_golden] sample: Should Zotiraciclib be used for glioblastoma?

[training13b] sample: Is Hirschsprung disease a mendelian or a multifactorial disorder?


---
## 🔤 Step 5: Tokenization

In [7]:
# Load a pretrained tokenizer
TOKENIZER_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_doc(doc: dict, max_length: int = 1024) -> dict:
    tokens = tokenizer(
        doc["text"],
        truncation=True,
        max_length=max_length,
        return_attention_mask=False,
    )
    return {
        **doc,
        "input_ids": tokens["input_ids"],
        "token_count": len(tokens["input_ids"]),
    }

print("⏳ Tokenizing training raw documents...")
tokenized_training_raw_data = [tokenize_doc(doc) for doc in tqdm(training_raw_data)]

total_tokens = sum(d["token_count"] for d in tokenized_training_raw_data)
avg_tokens = total_tokens / max(len(tokenized_training_raw_data), 1)
print(f"\n✅ Tokenized {len(tokenized_training_raw_data)} documents")
print(f"   Total tokens : {total_tokens:,}")
print(f"   Avg tokens/doc: {avg_tokens:.1f}")
print("")

print("⏳ Tokenizing test raw documents...")
tokenized_test_raw_data = [tokenize_doc(doc) for doc in tqdm(test_raw_data)]

total_tokens = sum(d["token_count"] for d in tokenized_test_raw_data)
avg_tokens = total_tokens / max(len(tokenized_test_raw_data), 1)
print(f"\n✅ Tokenized {len(tokenized_test_raw_data)} documents")
print(f"   Total tokens : {total_tokens:,}")
print(f"   Avg tokens/doc: {avg_tokens:.1f}")

⏳ Tokenizing training raw documents...


100% 4793/4793 [00:00<00:00, 16763.58it/s]



✅ Tokenized 4793 documents
   Total tokens : 69,131
   Avg tokens/doc: 14.4

⏳ Tokenizing test raw documents...


100% 295/295 [00:00<00:00, 14960.10it/s]


✅ Tokenized 295 documents
   Total tokens : 4,282
   Avg tokens/doc: 14.5


In [8]:
# break down of factoid vs list vs yesno vs summary in test set
test_types = [doc["type"] for doc in tokenized_test_raw_data]
type_counts = pd.Series(test_types).value_counts()
print("\n📊 Test set question type distribution:"   )
print(type_counts)  

# break down of factoid vs list vs yesno vs summary in training set
test_types = [doc["type"] for doc in tokenized_training_raw_data]
type_counts = pd.Series(test_types).value_counts()
print("\n📊 Training set question type distribution:"   )
print(type_counts)  


📊 Test set question type distribution:
factoid    88
yesno      78
list       69
summary    60
Name: count, dtype: int64

📊 Training set question type distribution:
factoid    1497
yesno      1333
list       1002
summary     961
Name: count, dtype: int64
